In [ ]:
import random, math, numpy as np
from typing import List
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torch.nn.utils import spectral_norm
from tqdm.auto import tqdm
import plotly.graph_objects as go

SEED        = 40
EPOCHS      = 50
BATCH_SIZE  = 2048
LATENT_DIM  = 32
LR          = 1e-4
BETA        = 0.1
ALPHA_CE    = 1.0

DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 0
PERSISTENT  = False
PIN_MEMORY  = (DEVICE == "cuda")

MARGIN        = 1.0
TAU_MAX       = 1.0
N_TAU         = 21

SIGMA_Q       = 0.7
ALPHA         = 0.2
CLS_SCALE     = 0.1

SIGMA_MIN_GRID = 0.0
SIGMA_MAX_GRID = 1.0
N_SIGMA_GRID   = 11
SIGMA_GRID     = np.linspace(SIGMA_MIN_GRID, SIGMA_MAX_GRID, N_SIGMA_GRID)

def set_seed(seed: int = 43):
    random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

class Encoder(nn.Module):
    def __init__(self, latent_dim: int = 32):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 4, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=1, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
        )
        self.flat = nn.Flatten()
        self.fc_mu = nn.Linear(128*7*7, latent_dim)
        self.fc_logvar = nn.Linear(128*7*7, latent_dim)

    def forward(self, x):
        h = self.flat(self.conv(x))
        return self.fc_mu(h), self.fc_logvar(h)

class Decoder(nn.Module):
    def __init__(self, latent_dim: int = 32):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 128*7*7)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 3, padding=1),
        )

    def forward(self, z):
        h = self.fc(z).view(-1,128,7,7)
        x_hat = self.deconv(h)
        return torch.sigmoid(x_hat)

class VAE(nn.Module):
    def __init__(self, latent_dim: int = 32, num_classes: int = 10):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)
        self.classifier = spectral_norm(nn.Linear(latent_dim, num_classes))

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5*logvar); eps = torch.randn_like(std)
        return mu + eps*std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decoder(z)
        logits = CLS_SCALE * self.classifier(mu)
        return x_recon, mu, logvar, logits

    @torch.no_grad()
    def logits_from_x_nograd(self, x):
        mu, _ = self.encoder(x)
        return CLS_SCALE * self.classifier(mu)

def kl_divergence(mu, logvar):
    return -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1).mean()

def reconstruction_loss_mse(x_recon, x):
    return F.mse_loss(x_recon, x, reduction='mean')

def cross_entropy_from_logits(logits, y):
    return F.cross_entropy(logits, y, reduction='mean')

def margin_hinge_loss_from_logits(logits, targets, margin=1.0):
    idx = torch.arange(logits.size(0), device=logits.device)
    masked = logits.clone()
    masked[idx, targets] = -float('inf')
    max_other = masked.max(dim=1).values
    m = logits[idx, targets] - max_other
    v = F.relu(margin - m)
    return v.mean()

MNIST_MEAN = (0.1307,)
MNIST_STD  = (0.3081,)

transform_train = transforms.Compose([
    transforms.RandomAffine(degrees=10, translate=(0.1,0.1), scale=(0.9,1.1)),
    transforms.ToTensor(),
    transforms.Normalize(MNIST_MEAN, MNIST_STD),
])

transform_test  = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MNIST_MEAN, MNIST_STD),
])

train_set = datasets.MNIST(root='./data_mnist', train=True,  download=True, transform=transform_train)
test_set  = datasets.MNIST(root='./data_mnist', train=False, download=True, transform=transform_test)

def make_half_splits(dataset, batch_size=128):
    n = len(dataset)
    idx_cal = np.arange(0, n//2)
    idx_tst = np.arange(n//2, n)
    ds_cal  = Subset(dataset, idx_cal)
    ds_tst  = Subset(dataset, idx_tst)
    ld_cal  = DataLoader(ds_cal, batch_size=batch_size, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT)
    ld_tst  = DataLoader(ds_tst, batch_size=batch_size, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT)
    return ld_cal, ld_tst

P_cal_loader, P_tst_loader = make_half_splits(test_set, BATCH_SIZE)
Q_cal_loader, Q_tst_loader = P_cal_loader, P_tst_loader

def apply_Q_transform(x: torch.Tensor, sigma: float) -> torch.Tensor:
    if sigma <= 0.0:
        return x

    B = x.size(0)
    device = x.device

    contrast_scale = 1.0 + 0.8 * sigma * (2.0 * torch.rand(B, 1, 1, 1, device=device) - 1.0)
    brightness_shift = 0.4 * sigma * (2.0 * torch.rand(B, 1, 1, 1, device=device) - 1.0)
    x = x * contrast_scale + brightness_shift

    max_shift = int(4 * sigma)
    if max_shift > 0:
        shift_x = torch.randint(-max_shift, max_shift + 1, (B,), device=device)
        shift_y = torch.randint(-max_shift, max_shift + 1, (B,), device=device)
        for i in range(B):
            x[i] = torch.roll(x[i], shifts=(int(shift_y[i]), int(shift_x[i])), dims=(1, 2))

    noise_std = 0.1 * sigma
    if noise_std > 0:
        x = x + noise_std * torch.randn_like(x)

    return x

model = VAE(latent_dim=LATENT_DIM, num_classes=10).to(DEVICE)
params_classifier = list(model.classifier.parameters())
params_vae = [p for n, p in model.named_parameters() if not n.startswith("classifier.")]
optimizer = torch.optim.Adam(
    [{'params': params_vae, 'weight_decay': 0.0},
     {'params': params_classifier, 'weight_decay': 0.0}],
    lr=LR
)

def train_one_epoch(epoch: int):
    model.train()
    sums = {'recon':0.0,'kl':0.0,'cce':0.0,'hinge':0.0,'acc':0.0,'n':0}
    pbar = tqdm(DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT),
                desc=f"Train {epoch:02d}")
    for x, y in pbar:
        x, y = x.to(DEVICE), y.to(DEVICE)
        x_recon, mu, logvar, logits = model(x)
        recon = reconstruction_loss_mse(x_recon, x)
        kld   = kl_divergence(mu, logvar)
        cce   = cross_entropy_from_logits(logits, y)
        hinge = margin_hinge_loss_from_logits(logits, y, margin=MARGIN)
        loss = recon + BETA * kld + ALPHA_CE * cce + 5.0 * hinge

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            acc = (logits.argmax(1) == y).float().mean().item()
        bs = x.size(0)
        sums['recon'] += recon.item()*bs
        sums['kl']    += kld.item()*bs
        sums['cce']   += cce.item()*bs
        sums['hinge'] += hinge.item()*bs
        sums['acc']   += acc*bs
        sums['n']     += bs
        pbar.set_postfix({k:f"{sums[k]/sums['n']:.4f}" for k in ['recon','kl','cce','hinge','acc']})
    for k in ['recon','kl','cce','hinge','acc']:
        sums[k] /= sums['n']
    return sums

@torch.no_grad()
def evaluate(loader, tag="Eval"):
    model.eval()
    tot = {'cce':0.0,'hinge':0.0,'acc':0.0,'n':0}
    pbar = tqdm(loader, desc=f"{tag}", leave=False)
    for x, y in pbar:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model.logits_from_x_nograd(x)
        cce   = F.cross_entropy(logits, y, reduction='mean').item()
        hinge = margin_hinge_loss_from_logits(logits, y, margin=MARGIN).item()
        acc   = (logits.argmax(1) == y).float().mean().item()
        bs = x.size(0)
        tot['cce']+=cce*bs; tot['hinge']+=hinge*bs; tot['acc']+=acc*bs; tot['n']+=bs
    for k in ['cce','hinge','acc']:
        tot[k] /= tot['n']
    return tot

@torch.no_grad()
def collect_scores_truey(model: VAE, loader: DataLoader, noise_sigma: float = 0.0) -> np.ndarray:
    model.eval(); out = []
    for x, y in tqdm(loader, desc=f"Scores(true y) σ={noise_sigma}", leave=False):
        x = x.to(DEVICE); y = y.to(DEVICE)
        if noise_sigma > 0.0:
            x = apply_Q_transform(x, float(noise_sigma))
        logits = model.logits_from_x_nograd(x)
        idx = torch.arange(logits.size(0), device=logits.device)
        masked = logits.clone(); masked[idx, y] = -float('inf')
        max_other = masked.max(dim=1).values
        s = (max_other - logits[idx, y]).detach().float().cpu().numpy()
        out.append(s)
    return np.concatenate(out, axis=0) if out else np.array([])

@torch.no_grad()
def collect_scores_pseudoy(model: VAE, loader: DataLoader, noise_sigma: float = 0.0) -> np.ndarray:
    model.eval(); out = []
    for x, _ in tqdm(loader, desc=f"Scores(pseudo y) σ={noise_sigma}", leave=False):
        x = x.to(DEVICE)
        if noise_sigma > 0.0:
            x = apply_Q_transform(x, float(noise_sigma))
        logits = model.logits_from_x_nograd(x)
        yhat = logits.argmax(1)
        idx = torch.arange(logits.size(0), device=logits.device)
        masked = logits.clone(); masked[idx, yhat] = -float('inf')
        max_other = masked.max(dim=1).values
        s = (max_other - logits[idx, yhat]).detach().float().cpu().numpy()
        out.append(s)
    return np.concatenate(out, axis=0) if out else np.array([])

def conformal_quantile(scores: np.ndarray, alpha: float, finite_sample: bool = True) -> float:
    n = scores.size
    if n == 0:
        return float('nan')
    if finite_sample:
        qlevel = math.ceil((n + 1) * (1 - alpha)) / n
    else:
        qlevel = 1 - alpha
    if qlevel > 1:
        return float('inf')
    try:
        return float(np.quantile(scores, qlevel, method='higher'))
    except TypeError:
        return float(np.quantile(scores, qlevel, interpolation='higher'))

def coverage_given_threshold(scores: np.ndarray, threshold: float) -> float:
    if scores.size == 0:
        return float('nan')
    return float((scores <= threshold).mean())

TAU_GRID = np.linspace(0.0, TAU_MAX, N_TAU)

hinge_P_epochs      = []
cov_Q_self_epochs   = []
cov_P_src_epochs    = []
qP_epochs           = []
qQ_epochs           = []

qQ_epochs_sigma     = np.zeros((EPOCHS, N_SIGMA_GRID), dtype=np.float64)

cov_Q_sigma0_epochs = np.zeros((EPOCHS, N_SIGMA_GRID), dtype=np.float64)
cov_P_sigma0_epochs = np.zeros((EPOCHS, N_SIGMA_GRID), dtype=np.float64)

print("=== Training with per-epoch coverage tracking on MNIST (σ-grid, τ-grid) ===")
for epoch in range(1, EPOCHS+1):
    tr = train_one_epoch(epoch)

    teP = evaluate(P_tst_loader, tag=f"Test(P) epoch {epoch}")
    hinge_P = teP["hinge"]
    hinge_P_epochs.append(hinge_P)

    scores_P_cal_true_0_e = collect_scores_truey(
        model, P_cal_loader, noise_sigma=0.0
    )
    q_P_e = conformal_quantile(scores_P_cal_true_0_e, ALPHA, finite_sample=True)
    qP_epochs.append(q_P_e)

    q_Q_SIGMAQ_e = None
    scores_Q_tst_true_SIGMAQ = None

    for s_idx, sigma in enumerate(SIGMA_GRID):
        sigma_val = float(sigma)

        scores_Q_cal_pseudo_sigma = collect_scores_pseudoy(
            model, Q_cal_loader, noise_sigma=sigma_val
        )
        q_Q_sigma_e = conformal_quantile(
            scores_Q_cal_pseudo_sigma, ALPHA, finite_sample=True
        )
        qQ_epochs_sigma[epoch-1, s_idx] = q_Q_sigma_e

        scores_Q_tst_true_sigma = collect_scores_truey(
            model, Q_tst_loader, noise_sigma=sigma_val
        )

        cov_Q_sigma0_epochs[epoch-1, s_idx] = coverage_given_threshold(
            scores_Q_tst_true_sigma, q_Q_sigma_e
        )
        cov_P_sigma0_epochs[epoch-1, s_idx] = coverage_given_threshold(
            scores_Q_tst_true_sigma, q_P_e
        )

        if abs(sigma_val - SIGMA_Q) < 1e-8:
            q_Q_SIGMAQ_e = q_Q_sigma_e
            scores_Q_tst_true_SIGMAQ = scores_Q_tst_true_sigma

    if q_Q_SIGMAQ_e is None or scores_Q_tst_true_SIGMAQ is None:
        raise RuntimeError("SIGMA_Q is not in SIGMA_GRID; adjust SIGMA_GRID to include SIGMA_Q.")

    qQ_epochs.append(q_Q_SIGMAQ_e)

    cov_Q_self = [
        coverage_given_threshold(scores_Q_tst_true_SIGMAQ, q_Q_SIGMAQ_e + t)
        for t in TAU_GRID
    ]
    cov_P_src = [
        coverage_given_threshold(scores_Q_tst_true_SIGMAQ, q_P_e + t)
        for t in TAU_GRID
    ]

    cov_Q_self_epochs.append(cov_Q_self)
    cov_P_src_epochs.append(cov_P_src)

    print(
        f"Epoch {epoch:02d} | "
        f"Train CE {tr['cce']:.4f} hinge {tr['hinge']:.4f} acc {tr['acc']:.4f} | "
        f"Test(P) CE {teP['cce']:.4f} hinge {teP['hinge']:.4f} acc {teP['acc']:.4f} | "
        f"q_P={q_P_e:.4f}, q̂_Q(SIGMA_Q)={q_Q_SIGMAQ_e:.4f}"
    )

hinge_P_epochs    = np.array(hinge_P_epochs, dtype=np.float64)
cov_Q_self_epochs = np.array(cov_Q_self_epochs, dtype=np.float64)
cov_P_src_epochs  = np.array(cov_P_src_epochs, dtype=np.float64)
qP_epochs         = np.array(qP_epochs, dtype=np.float64)
qQ_epochs         = np.array(qQ_epochs, dtype=np.float64)

print("\nPer-epoch hinge loss on P_tst:", hinge_P_epochs)
print("Per-epoch q_P:", qP_epochs)
print("Per-epoch q̂_Q(SIGMA_Q):", qQ_epochs)

Z_self = cov_Q_self_epochs
Z_src  = cov_P_src_epochs
nominal_cov = 1.0 - ALPHA
Z_nom  = nominal_cov * np.ones_like(Z_self)

fig3d_cov = go.Figure()

fig3d_cov.add_surface(
    x=TAU_GRID,
    y=hinge_P_epochs,
    z=Z_self,
    colorscale="Viridis",
    opacity=0.9,
    name="Self-Cal Q→Q (q̂_Q)",
    showscale=False,
)

fig3d_cov.add_surface(
    x=TAU_GRID,
    y=hinge_P_epochs,
    z=Z_src,
    colorscale="Plasma",
    opacity=0.7,
    name="Source-Cal P→Q (q_P)",
    showscale=False,
)

fig3d_cov.add_surface(
    x=TAU_GRID,
    y=hinge_P_epochs,
    z=Z_nom,
    colorscale=[[0, "lightgrey"], [1, "lightgrey"]],
    opacity=0.3,
    name="Nominal 1−α plane",
    showscale=False,
)

fig3d_cov.update_layout(
    title=(
        f"MNIST: Coverage vs (τ, hinge loss on P)<br>"
        f"Self-Cal Q→Q (q̂_Q), Source-Cal P→Q (q_P), "
        f"σ_Q={SIGMA_Q}, α={ALPHA}"
    ),
    scene=dict(
        xaxis_title="τ (score inflation)",
        yaxis_title="Hinge loss on P (P_tst)",
        zaxis_title="Coverage on Q",
        zaxis=dict(range=[0.0, 1.02]),
    ),
    width=900,
    height=700,
)

fig3d_cov.write_html("mnist_coverage_vs_tau_hinge_3d.html", include_plotlyjs="cdn")
print("Saved 3D coverage plot to mnist_coverage_vs_tau_hinge_3d.html")

Z_diff = Z_self - Z_src

fig3d_covdiff_tau = go.Figure()

fig3d_covdiff_tau.add_surface(
    x=TAU_GRID,
    y=hinge_P_epochs,
    z=Z_diff,
    colorscale="RdBu",
    opacity=0.9,
    name="Coverage diff (Self - Source)",
    showscale=True,
)

fig3d_covdiff_tau.add_surface(
    x=TAU_GRID,
    y=hinge_P_epochs,
    z=np.zeros_like(Z_diff),
    colorscale=[[0, "black"], [1, "black"]],
    opacity=0.2,
    name="Zero plane",
    showscale=False,
)

fig3d_covdiff_tau.update_layout(
    title=(
        f"MNIST: Coverage difference vs (τ, hinge loss on P)<br>"
        f"(Self-Cal Q→Q − Source-Cal P→Q), σ_Q={SIGMA_Q}, α={ALPHA}"
    ),
    scene=dict(
        xaxis_title="τ (score inflation)",
        yaxis_title="Hinge loss on P (P_tst)",
        zaxis_title="Coverage difference",
    ),
    width=900,
    height=700,
)

fig3d_covdiff_tau.write_html("mnist_covdiff_vs_tau_hinge_3d.html", include_plotlyjs="cdn")
print("Saved 3D coverage-difference (τ,hinge) plot to mnist_covdiff_vs_tau_hinge_3d.html")

Z_diff_sigma = (cov_Q_sigma0_epochs - cov_P_sigma0_epochs).T

LOSS_AXIS  = hinge_P_epochs
SIGMA_AXIS = SIGMA_GRID

LOSS_MESH, SIGMA_MESH = np.meshgrid(LOSS_AXIS, SIGMA_AXIS)

fig3d_covdiff_sigma = go.Figure()

fig3d_covdiff_sigma.add_surface(
    x=LOSS_MESH,
    y=SIGMA_MESH,
    z=Z_diff_sigma,
    colorscale="RdBu",
    opacity=0.9,
    name="Coverage diff (Self - Source) at τ=0",
    showscale=True,
)

fig3d_covdiff_sigma.add_surface(
    x=LOSS_MESH,
    y=SIGMA_MESH,
    z=np.zeros_like(Z_diff_sigma),
    colorscale=[[0, "black"], [1, "black"]],
    opacity=0.2,
    name="Zero plane",
    showscale=False,
)

fig3d_covdiff_sigma.update_layout(
    title=(
        "MNIST: Coverage difference vs (hinge loss, σ) at τ=0<br>"
        "(Self-Cal Q→Q − Source-Cal P→Q)"
    ),
    scene=dict(
        xaxis_title="Hinge loss on P (P_tst)",
        yaxis_title="σ (Q shift level)",
        zaxis_title="Coverage difference",
    ),
    width=900,
    height=700,
)

fig3d_covdiff_sigma.write_html("mnist_covdiff_vs_loss_sigma_tau0_3d.html", include_plotlyjs="cdn")
print("Saved 3D coverage-difference (loss,σ, τ=0) plot to mnist_covdiff_vs_loss_sigma_tau0_3d.html")

Z_Q_sigma = qQ_epochs_sigma.T
Z_P_sigma = np.tile(qP_epochs, (N_SIGMA_GRID, 1))

fig3d_q = go.Figure()

fig3d_q.add_surface(
    x=LOSS_MESH,
    y=SIGMA_MESH,
    z=Z_Q_sigma,
    colorscale="Viridis",
    opacity=0.9,
    name="q̂_Q(σ) (Q_cal, pseudo)",
    showscale=False,
)

fig3d_q.add_surface(
    x=LOSS_MESH,
    y=SIGMA_MESH,
    z=Z_P_sigma,
    colorscale=[[0, "lightgrey"], [1, "lightgrey"]],
    opacity=0.4,
    name="q_P (P_cal, σ=0)",
    showscale=False,
)

fig3d_q.update_layout(
    title=(
        "MNIST: Quantiles vs (hinge loss, σ)<br>"
        "Self-Cal q̂_Q(σ) and Source-Cal q_P"
    ),
    scene=dict(
        xaxis_title="Hinge loss on P (P_tst)",
        yaxis_title="σ (Q shift level)",
        zaxis_title="Quantile value",
    ),
    width=900,
    height=700,
)

fig3d_q.write_html("mnist_quantiles_vs_loss_sigma_3d.html", include_plotlyjs="cdn")
print("Saved 3D quantile surface to mnist_quantiles_vs_loss_sigma_3d.html")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import scienceplots
import matplotlib as mpl

plt.style.use(["science", "ieee", "no-latex"])

mpl.rcParams["font.size"] = 12
mpl.rcParams["axes.labelsize"] = 12
mpl.rcParams["axes.titlesize"] = 12
mpl.rcParams["legend.fontsize"] = 11
mpl.rcParams["xtick.labelsize"] = 11
mpl.rcParams["ytick.labelsize"] = 11

mpl.rcParams["axes.unicode_minus"] = False
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

IEEE_WIDTH_1COL = 3.5
IEEE_ASPECT = 0.9
FIGSIZE_1COL = (IEEE_WIDTH_1COL, IEEE_WIDTH_1COL * IEEE_ASPECT)

ELEV = 30
AZIM = -60

CMAP = "coolwarm"

TAU, HINGE = np.meshgrid(TAU_GRID, hinge_P_epochs)

LOSS_AXIS  = hinge_P_epochs
SIGMA_AXIS = SIGMA_GRID
LOSS_MESH, SIGMA_MESH = np.meshgrid(LOSS_AXIS, SIGMA_AXIS)

Z_self       = cov_Q_self_epochs
Z_src        = cov_P_src_epochs
Z_nom        = (1.0 - ALPHA) * np.ones_like(Z_self)
Z_diff_tau   = Z_self - Z_src

Z_diff_sigma = (cov_Q_sigma0_epochs - cov_P_sigma0_epochs).T
Z_Q_sigma    = qQ_epochs_sigma.T
Z_P_sigma    = np.tile(qP_epochs, (N_SIGMA_GRID, 1))

Z_diff_tau_pct   = 100.0 * Z_diff_tau
Z_diff_sigma_pct = 100.0 * Z_diff_sigma

fig = plt.figure(figsize=FIGSIZE_1COL)
ax = fig.add_subplot(111, projection="3d")

ax.plot_surface(
    TAU, HINGE, Z_self,
    alpha=0.9, linewidth=0, antialiased=True, cmap=CMAP
)
ax.plot_surface(
    TAU, HINGE, Z_src,
    alpha=0.6, linewidth=0, antialiased=True, cmap=CMAP
)
ax.plot_surface(
    TAU, HINGE, Z_nom,
    alpha=0.3, linewidth=0, antialiased=True, cmap=CMAP
)

ax.view_init(elev=ELEV, azim=AZIM)

ax.set_xlabel(r"$\tau$ (score inflation)")
ax.set_ylabel("Hinge loss on $P$")
ax.set_zlabel("Coverage on $Q$")
ax.set_zlim(0.0, 1.02)

fig.tight_layout()
fig.savefig("mnist_coverage_vs_tau_hinge_3d_science.pdf", bbox_inches="tight")
plt.close(fig)

fig = plt.figure(figsize=FIGSIZE_1COL)
ax = fig.add_subplot(111, projection="3d")

ax.plot_surface(
    TAU, HINGE, Z_diff_tau_pct,
    alpha=0.9, linewidth=0, antialiased=True, cmap=CMAP
)
ax.plot_surface(
    TAU, HINGE, np.zeros_like(Z_diff_tau_pct),
    alpha=0.2, linewidth=0, antialiased=True, cmap=CMAP
)

ax.view_init(elev=ELEV, azim=AZIM)

ax.set_xlabel(r"$\tau$ (score inflation)")
ax.set_ylabel("Hinge loss on $P$")
ax.set_zlabel("Coverage difference (%)")

fig.tight_layout()
fig.savefig("mnist_covdiff_vs_tau_hinge_3d_science.pdf", bbox_inches="tight")
plt.close(fig)

fig = plt.figure(figsize=FIGSIZE_1COL)
ax = fig.add_subplot(111, projection="3d")

ax.plot_surface(
    LOSS_MESH, SIGMA_MESH, Z_diff_sigma_pct,
    alpha=0.9, linewidth=0, antialiased=True, cmap=CMAP
)
ax.plot_surface(
    LOSS_MESH, SIGMA_MESH, np.zeros_like(Z_diff_sigma_pct),
    alpha=0.2, linewidth=0, antialiased=True, cmap=CMAP
)

ax.view_init(elev=ELEV, azim=AZIM)

ax.set_xlabel("Hinge loss on $P$")
ax.set_ylabel(r"$\sigma$ (Q shift level)")
ax.set_zlabel(r"Coverage difference at $\tau=0$ (%)")

fig.tight_layout()
fig.savefig("mnist_covdiff_vs_loss_sigma_tau0_3d_science.pdf", bbox_inches="tight")
plt.close(fig)

fig = plt.figure(figsize=FIGSIZE_1COL)
ax = fig.add_subplot(111, projection="3d")

ax.plot_surface(
    LOSS_MESH, SIGMA_MESH, Z_Q_sigma,
    alpha=0.9, linewidth=0, antialiased=True, cmap=CMAP
)
ax.plot_surface(
    LOSS_MESH, SIGMA_MESH, Z_P_sigma,
    alpha=0.4, linewidth=0, antialiased=True, cmap=CMAP
)

ax.view_init(elev=ELEV, azim=AZIM)

ax.set_xlabel("Hinge loss on $P$")
ax.set_ylabel(r"$\sigma$ (Q shift level)")
ax.set_zlabel("Quantile")

fig.tight_layout()
fig.savefig("mnist_quantiles_vs_loss_sigma_3d_science.pdf", bbox_inches="tight")
plt.close(fig)

print("Saved IEEE-style 1-column scienceplots PDFs with coolwarm colormap (coverage diffs in %).")